# Caso F · 03 Reproducibilidad — hash dataset, hash modelo

> _Tutorial · Caso de uso: **F — MLOps** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Demostrar reproducibilidad bit-a-bit usando hashes SHA-256 sobre el dataset y un identificador (params + seed) del modelo.


## 2. Qué se aprende

- Por qué hash de fichero != hash de DataFrame.
- Cómo simular un lakeFS tag con un hash local.
- Cuándo se rompe la reproducibilidad.


## 3. Contexto del caso de uso

Mismo seed + mismos datos → mismo modelo. La auditoría debe poder verificarlo automáticamente.


## 4. Relación con CENTINELA+

Idéntico.


## 5. Relación con Medallion

Transversal.


## 6. Datos de entrada

Mock BDG2.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos.


In [2]:
df, _ = mocks.make_bdg2_education_subset()
print(df.shape)


(51840, 5)


## 10. Exploración paso a paso

Hashing.


In [3]:
import hashlib

def df_hash(df: pd.DataFrame) -> str:
    rep = df.sort_index(axis=1).to_csv(index=False).encode()
    return hashlib.sha256(rep).hexdigest()

h1 = df_hash(df)
df_again, _ = mocks.make_bdg2_education_subset()
h2 = df_hash(df_again)
print("hash 1:", h1[:16])
print("hash 2:", h2[:16])
assert h1 == h2, "Reproducibilidad rota"


hash 1: a4c97d146796f33f
hash 2: a4c97d146796f33f


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Modelo + tag.


In [4]:
from sklearn.ensemble import RandomForestRegressor

X = pd.DataFrame({
    "y": df["power_kw"], "t": df["t_outdoor"], "ghi": df["ghi"],
}).dropna()
y = X.pop("y")
m1 = RandomForestRegressor(n_estimators=50, random_state=SEED).fit(X, y)
m2 = RandomForestRegressor(n_estimators=50, random_state=SEED).fit(X, y)
import joblib, io
b1 = io.BytesIO(); joblib.dump(m1, b1)
b2 = io.BytesIO(); joblib.dump(m2, b2)
hm1 = hashlib.sha256(b1.getvalue()).hexdigest()
hm2 = hashlib.sha256(b2.getvalue()).hexdigest()
print(hm1[:16], hm2[:16])


c49b287e5410b15e c49b287e5410b15e


## 13. Visualizaciones explicativas

Tabla resumen.


In [5]:
print(pd.DataFrame({"obj": ["dataset", "model"], "hash": [h1[:16], hm1[:16]]}))


       obj              hash
0  dataset  a4c97d146796f33f
1    model  c49b287e5410b15e


## 14. Validaciones

Reproducibilidad estricta del dataset y aproximada del modelo (joblib lleva metadatos de hora).


In [6]:
assert h1 == h2


## 15. Errores comunes

1. Hash con orden de columnas diferente.
2. Hash incluyendo metadata de joblib (hora) — usar pickle puro.
3. No fijar el seed.


## 16. Ejercicios propuestos

1. Implementa una función que combine hash dataset + params en un único id.
2. Discute por qué pyarrow puede romper la reproducibilidad.
3. Diseña un workflow de PR-checking que use estos hashes.


## 17. Cómo se reutiliza con datos reales

Mismo principio; lakeFS se encarga del hashing en producción.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `07_case_G_data_quality_agents/01_reglas_calidad_bronce.ipynb`.
- Documento web del caso: `docs/use-cases/case-f-mlops.md`.


## 19. Marco teórico (nivel doctoral)

### Versionado de datasets (lakeFS)

Modelo Git-like sobre object storage con commits inmutables:

$$
\text{commit}_t = \langle \text{tree}_t, \text{parent}_{t-1}, \text{metadata}_t \rangle
$$

con `tree` Merkle DAG. Modelos referencian
$\text{dataset\_uri} = \text{lakefs://repo/branch/commit\_id}$ no paths mutables.

### MLflow Run Schema

$$
\text{run}_i = \langle \text{params}_i, \text{metrics}_i, \text{artifacts}_i, \text{tags}_i, \text{dataset\_uri}_i \rangle
$$

### Reproducibilidad determinista

$$
\hat{y} = M(D, \theta, s = 42), \quad
\text{hash}(\hat{y}_1) = \text{hash}(\hat{y}_2) \iff (D_1, \theta_1, s_1) = (D_2, \theta_2, s_2)
$$

(ADR-008). Verificable con `pytest -m snapshot`.

### Linaje de features

$$
\text{Feature}_i = f_i(\text{Feature}_j, \text{Feature}_k, ...) \implies \text{DAG}_{features}
$$

Trazabilidad bidireccional dataset $\leftrightarrow$ run $\leftrightarrow$ deploy.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

MLOps no genera ROI directo, pero **reduce el coste de toda la cadena**. Permite a CAPTIA mover modelos a producción con confianza, hacer auditorías regulatorias (próxima EU AI Act) y evitar el clásico *funcionaba en mi máquina*.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tiempo auditoría modelos (8 h → 1 h) | +800 €/año |
| Eliminación re-runs por incertidumbre | +400 €/año compute |
| Cumplimiento EU AI Act (riesgo evitado) | +20 000 € (riesgo) |
| **Bruto** | **+1 200 €/año** + risk hedging |


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2018). *Accelerating the Machine Learning Lifecycle with MLflow*. CIDR.
- lakeFS Project. *Documentation*. https://docs.lakefs.io
- Sculley, D. et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
- EU (2024). *Artificial Intelligence Act*. Regulation (EU) 2024/1689.
